In [6]:
!pip install streamlit pyngrok pandas matplotlib


In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import re
import random
import json
import matplotlib.pyplot as plt
from datetime import datetime

# ==================================================
# PART 1: Synthetic Breach Dataset (Public / Safe)
# ==================================================
breach_data = [
    {"email": "john@gmail.com", "breach_status": "Yes", "breached_by": "DarkWeb Group", "website": "Forum"},
    {"email": "student@spit.ac.in", "breach_status": "Yes", "breached_by": "DataLeak Corp", "website": "Education"},
    {"email": "alice@yahoo.com", "breach_status": "No", "breached_by": "None", "website": "None"},
    {"email": "employee@bank.com", "breach_status": "Yes", "breached_by": "Unknown Hacker", "website": "Finance"},
    {"email": "learner@spit", "breach_status": "Yes", "breached_by": "DarkWeb Group", "website": "Education"}
]

breach_df = pd.DataFrame(breach_data)

# ==================================================
# PART 2: Synthetic Leak Files (for PII detection)
# ==================================================
leak_files = [
    {"source": "dump1.csv", "content": "John Doe john@gmail.com 9876543210"},
    {"source": "dump2.csv", "content": "Student One student@spit.ac.in"},
    {"source": "dump3.csv", "content": "Alice alice@yahoo.com"},
    {"source": "dump4.csv", "content": "Employee emp@bank.com 9123456789"},
    {"source": "dump5.csv", "content": "Learner learner@spit"}
]

leak_df = pd.DataFrame(leak_files)

# ==================================================
# PART 3: Regex-based PII Detection
# ==================================================
email_regex = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
phone_regex = r"\b[6-9][0-9]{9}\b"
name_regex = r"\b(John|Doe|Alice|Student|Employee|Learner)\b"

alerts = []

def confidence(pii):
    return {"Email": 0.95, "Phone": 0.90, "Name": 0.80}.get(pii, 0.75)

for _, row in leak_df.iterrows():
    content = row["content"]

    for e in re.findall(email_regex, content):
        alerts.append({
            "source": row["source"],
            "pii_type": "Email",
            "value": e,
            "institutional": e.endswith("@spit") or e.endswith("@spit.ac.in") or e.endswith(".ac.in"),
            "confidence": confidence("Email"),
            "timestamp": datetime.now().isoformat()
        })

    for p in re.findall(phone_regex, content):
        alerts.append({
            "source": row["source"],
            "pii_type": "Phone",
            "value": p,
            "institutional": False,
            "confidence": confidence("Phone"),
            "timestamp": datetime.now().isoformat()
        })

    for n in re.findall(name_regex, content):
        alerts.append({
            "source": row["source"],
            "pii_type": "Name",
            "value": n,
            "institutional": False,
            "confidence": confidence("Name"),
            "timestamp": datetime.now().isoformat()
        })

alert_df = pd.DataFrame(alerts)

# Save reports
alert_df.to_csv("alert_report.csv", index=False)
with open("alert_report.json", "w") as f:
    json.dump(alerts, f, indent=4)

# ==================================================
# STREAMLIT DASHBOARD
# ==================================================
st.title("🔐 Email Breach & PII Detection Dashboard")

# -------------------------------
# Section 1: Email Breach Checker
# -------------------------------
st.header("📧 Check Breach Status")

user_email = st.text_input("Enter your email ID")

if st.button("Check Breach"):
    record = breach_df[breach_df["email"] == user_email]

    if user_email.endswith("@spit") or user_email.endswith("@spit.ac.in") or user_email.endswith(".ac.in"):
        st.warning("⚠ Institutional / SPIT email detected")

    if record.empty:
        st.error("Email not found in breach dataset")
    else:
        st.success("Breach Information Found")
        st.table(record)

# -------------------------------
# Section 2: PII Detection Alerts
# -------------------------------
st.header("🕵️ PII Detection Alerts")
st.dataframe(alert_df)

st.metric("Total PII Alerts", len(alert_df))
st.metric("Institutional Emails Detected", alert_df["institutional"].sum())

# -------------------------------
# Visualization
# -------------------------------
st.subheader("📊 PII Type Distribution")

fig, ax = plt.subplots()
alert_df["pii_type"].value_counts().plot(kind="bar", ax=ax)
st.pyplot(fig)

# -------------------------------
# Downloads
# -------------------------------
st.subheader("📥 Download Alert Reports")
st.download_button("Download CSV Report", open("alert_report.csv", "rb"), "alert_report.csv")
st.download_button("Download JSON Report", open("alert_report.json", "rb"), "alert_report.json")


Overwriting app.py


In [11]:
from pyngrok import ngrok

ngrok.set_auth_token("34vW4EWCNvd1rZ7Yc9xLTCGCEn3_7GA4PSKvnHFRH1uCQ7AgN")
public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://leafless-brandon-snugly.ngrok-free.dev" -> "http://localhost:8501">

In [12]:
!streamlit run app.py &>/content/logs.txt &
